# Movie Recommendation System
## Phase 4: Feature Engineering

In this notebook, we will transform movie genres into numerical
features using Multi-Hot Encoding.

We will also prepare the user-movie rating data for collaborative
filtering.

The purpose is to convert our raw data into formats that
recommendation algorithms can process.

In [1]:
import pandas as pd
import numpy as np

from pathlib import Path
from sklearn.preprocessing import MultiLabelBinarizer

In [2]:
DATA_DIR = Path("../data/processed")

movies = pd.read_csv(DATA_DIR / "movies_clean.csv")
ratings = pd.read_csv(DATA_DIR / "ratings_clean.csv")
users = pd.read_csv(DATA_DIR / "users_clean.csv")

print("Processed datasets loaded successfully!")

Processed datasets loaded successfully!


In [3]:
movies["GenreList"] = movies["Genres"].apply(
    lambda x: x.split("|")
)

movies[["Title", "Genres", "GenreList"]].head(10)

,Title,Genres,GenreList
0,Toy Story (1995),Animation|Children's|Comedy,"[Animation, Children's, Comedy]"
1,Jumanji (1995),Adventure|Children's|Fantasy,"[Adventure, Children's, Fantasy]"
2,Grumpier Old Men (1995),Comedy|Romance,"[Comedy, Romance]"
3,Waiting to Exhale (1995),Comedy|Drama,"[Comedy, Drama]"
4,Father of the Bride Part II (1995),Comedy,[Comedy]
5,Heat (1995),Action|Crime|Thriller,"[Action, Crime, Thriller]"
6,Sabrina (1995),Comedy|Romance,"[Comedy, Romance]"
7,Tom and Huck (1995),Adventure|Children's,"[Adventure, Children's]"
8,Sudden Death (1995),Action,[Action]
9,GoldenEye (1995),Action|Adventure|Thriller,"[Action, Adventure, Thriller]"


In [4]:
mlb = MultiLabelBinarizer()

genre_features = mlb.fit_transform(
    movies["GenreList"]
)

print("Genre feature matrix shape:", genre_features.shape)
print("Available genres:", mlb.classes_)

Genre feature matrix shape: (3883, 18)
Available genres: ['Action' 'Adventure' 'Animation' "Children's" 'Comedy' 'Crime'
 'Documentary' 'Drama' 'Fantasy' 'Film-Noir' 'Horror' 'Musical' 'Mystery'
 'Romance' 'Sci-Fi' 'Thriller' 'War' 'Western']


In [5]:
genre_df = pd.DataFrame(
    genre_features,
    columns=mlb.classes_,
    index=movies["MovieID"]
)

genre_df.head()

,Action,Adventure,Animation,Children's,Comedy,Crime,Documentary,Drama,Fantasy,Film-Noir,Horror,Musical,Mystery,Romance,Sci-Fi,Thriller,War,Western
MovieID,,,,,,,,,,,,,,,,,,
1,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,1,0,0,0,0,1,0,0,0,0,0,0,0,0,0
3,0,0,0,0,1,0,0,0,0,0,0,0,0,1,0,0,0,0
4,0,0,0,0,1,0,0,1,0,0,0,0,0,0,0,0,0,0
5,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0


In [6]:
print("Number of movies:", genre_df.shape[0])
print("Number of genre features:", genre_df.shape[1])

print("\nTotal genre assignments:", genre_df.sum().sum())

Number of movies: 3883
Number of genre features: 18

Total genre assignments: 6408


In [7]:
movie_id = movies.iloc[0]["MovieID"]

print(
    movies.loc[
        movies["MovieID"] == movie_id,
        ["MovieID", "Title", "Genres"]
    ]
)

print("\nEncoded features:")

print(genre_df.loc[movie_id])

   MovieID             Title                       Genres
0        1  Toy Story (1995)  Animation|Children's|Comedy

Encoded features:
Action         0
Adventure      0
Animation      1
Children's     1
Comedy         1
Crime          0
Documentary    0
Drama          0
Fantasy        0
Film-Noir      0
Horror         0
Musical        0
Mystery        0
Romance        0
Sci-Fi         0
Thriller       0
War            0
Western        0
Name: 1, dtype: int64


In [8]:
user_movie_matrix = ratings.pivot(
    index="UserID",
    columns="MovieID",
    values="Rating"
)

user_movie_matrix.head()

MovieID,1,2,3,4,5,6,7,8,9,10,...,3943,3944,3945,3946,3947,3948,3949,3950,3951,3952
UserID,,,,,,,,,,,,,,,,,,,,,
1,5.0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,2.0,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [9]:
print("Rating matrix shape:", user_movie_matrix.shape)
print("Total possible user-movie combinations:", user_movie_matrix.size)

print(
    "Actual rating records:",
    user_movie_matrix.notna().sum().sum()
)

Rating matrix shape: (6040, 3706)
Total possible user-movie combinations: 22384240
Actual rating records: 1000209


In [10]:
total_cells = user_movie_matrix.size

filled_cells = user_movie_matrix.notna().sum().sum()

sparsity = 1 - (filled_cells / total_cells)

print("Matrix sparsity:", round(sparsity * 100, 2), "%")

Matrix sparsity: 95.53 %


In [11]:
FEATURE_DIR = Path("../data/processed/features")
FEATURE_DIR.mkdir(parents=True, exist_ok=True)

genre_df.to_csv(
    FEATURE_DIR / "movie_genre_features.csv"
)

print("Genre features saved successfully!")

Genre features saved successfully!
